In [1]:
import h5py 
import numpy as np
import pandas as pd
import os
import scipy.signal as signal

In [2]:
def get_dataset_name(data):
    filename = data.split('/')[-1]
    print(f'filename: {filename}')
    temp = filename.split('_')[:-1]
    print(f'temp: {temp}')
    dataset_name = "_".join(temp)
    return dataset_name

In [5]:
file_path = '/Users/amancedennery/Downloads/Final Project data/Intra/train/rest_105923_1.h5'

with h5py.File(file_path, 'r')as f:
    dataset01 = get_dataset_name(file_path)
    matrix = f.get(dataset01)[()]
    print(type(matrix))
    print(matrix.shape)

filename: rest_105923_1.h5
temp: ['rest', '105923']
<class 'numpy.ndarray'>
(248, 35624)


In [26]:
tasks = ['rest', 'task_motor', 'task_story_math', 'task_working_memory']
# stats = {task: {'sum': np.zeros((248, 1)), 'sq_sum': np.zeros((248, 1)), 'steps': 0} for task in tasks}

def combine_train_arr():
    file_dir = r'/Users/amancedennery/Downloads/Final Project data/Intra/train/'
    nr = '_105923_'
    ini_arrays = {'rest': [], 'task_motor': [], 'task_story_math': [], 
                  'task_working_memory': []}    

    for task in tasks:
        for i in range(1, 9):
            #print(task+nr+str(i)+'.h5')
            file_path = os.path.join(file_dir, task+nr+str(i)+'.h5')
            

            with h5py.File(file_path, 'r')as f:
                file_keys = list(f.keys())
                if not file_keys:
                        print(f"Warning: File {file_path} is empty inside!")
                        continue
                
                dataset01 = file_keys[0]
                
                matrix = f.get(dataset01)[()]
                # print(type(matrix))
                # print(matrix.shape)
                ini_arrays[task].append(matrix)
    return ini_arrays

all_combined_arrs_intra = combine_train_arr()


In [8]:
def z_norm(matrix):
    mean = np.mean(matrix, axis = 1, keepdims = True)
    std = np.std(matrix, axis = 1, keepdims= True)
    matrix_z = (matrix - mean )/std
    
    return matrix_z


In [27]:
processed_datasets_intra = {}
for task_arr in all_combined_arrs_intra:
    raw_files = all_combined_arrs_intra[task_arr]
    downsampled_files = [downsampling_data(f, 4000) for f in raw_files]

    combined_task_data = np.concatenate(downsampled_files, axis =1)
    task_mean = np.mean(combined_task_data, axis=1, keepdims=True) # Shape: (248, 1)
    task_std = np.std(combined_task_data, axis=1, keepdims=True)  
    task_std = np.where(task_std == 0, 1.0, task_std) # Numerical stability guard
    

    normalized_files = []
    for ds_matrix in downsampled_files:
        norm_matrix = (ds_matrix - task_mean) / task_std
        normalized_files.append(norm_matrix)


    processed_datasets_intra[task_arr] = np.array(normalized_files)
    
    print(f"Task: {task_arr:<20} | Final clean shape: {processed_datasets_intra[task_arr].shape}")

Task: rest                 | Final clean shape: (8, 248, 4000)
Task: task_motor           | Final clean shape: (8, 248, 4000)
Task: task_story_math      | Final clean shape: (8, 248, 4000)
Task: task_working_memory  | Final clean shape: (8, 248, 4000)


In [29]:
tasks = ['rest', 'task_motor', 'task_story_math', 'task_working_memory']
# stats = {task: {'sum': np.zeros((248, 1)), 'sq_sum': np.zeros((248, 1)), 'steps': 0} for task in tasks}

def combine_cross_train_arr():
    file_dir = r'/Users/amancedennery/Downloads/Final Project data/Cross/train/'
    nrs = ["_113922_",  "_164636_"]
    ini_arrays = {'rest': [], 'task_motor': [], 'task_story_math': [], 
                  'task_working_memory': []}    

    for task in tasks:
        for nr in nrs:
            for i in range(1, 9):
                #print(task+nr+str(i)+'.h5')
                file_path = os.path.join(file_dir, task+nr+str(i)+'.h5')
                

                with h5py.File(file_path, 'r')as f:
                    file_keys = list(f.keys())
                    if not file_keys:
                            print(f"Warning: File {file_path} is empty inside!")
                            continue
                    
                    dataset01 = file_keys[0]
                    
                    matrix = f.get(dataset01)[()]
                    # print(type(matrix))
                    # print(matrix.shape)
                    ini_arrays[task].append(matrix)
    return ini_arrays

In [17]:
cross_combined_arr = combine_cross_train_arr()

In [18]:
x = cross_combined_arr.values()
print(x)

dict_values([[array([[ 3.29267061e-12,  3.25503752e-12,  3.13702796e-12, ...,
         3.58721386e-12,  3.71823232e-12,  3.88130847e-12],
       [-3.18692924e-12, -3.15210184e-12, -2.96682015e-12, ...,
        -3.33181768e-12, -3.54774738e-12, -3.82172289e-12],
       [ 1.58730494e-12,  1.42845815e-12,  1.32878962e-12, ...,
         2.00427695e-12,  2.00661297e-12,  2.03347669e-12],
       ...,
       [ 2.66890459e-12,  2.63538020e-12,  2.63910508e-12, ...,
         3.08349093e-12,  3.08721581e-12,  3.24105521e-12],
       [-4.54969049e-12, -4.60589249e-12, -4.64462626e-12, ...,
        -4.60246164e-12, -4.54398108e-12, -4.58043629e-12],
       [-3.75057450e-12, -3.75593914e-12, -3.69003222e-12, ...,
        -6.07841034e-12, -6.08109266e-12, -6.06844783e-12]]), array([[ 3.99234899e-12,  3.90546796e-12,  3.76655131e-12, ...,
         4.43232603e-12,  4.37285668e-12,  4.36495848e-12],
       [-3.88766276e-12, -3.66198131e-12, -3.44883779e-12, ...,
        -4.45925762e-12, -4.44300499e-12

In [28]:
processed_datasets_cross = {}
for task_arr in cross_combined_arr:
    raw_files = cross_combined_arr[task_arr]

    downsampled_files = [downsampling_data(f, 4000) for f in raw_files]
    
    combined_cross_task_data = np.concatenate(downsampled_files, axis =1)
    task_mean = np.mean(combined_cross_task_data, axis=1, keepdims=True) # Shape: (248, 1)
    task_std = np.std(combined_cross_task_data, axis=1, keepdims=True)   
    task_std = np.where(task_std == 0, 1.0, task_std) # Numerical stability guard

    normalized_files = []
    for ds_matrix in downsampled_files:
        norm_matrix = (ds_matrix - task_mean) / task_std
        normalized_files.append(norm_matrix)
        

    processed_datasets_cross[task_arr] = np.array(normalized_files)
    
    print(f"Task: {task_arr:<20} | Final clean shape: {processed_datasets_cross[task_arr].shape}")

Task: rest                 | Final clean shape: (16, 248, 4000)
Task: task_motor           | Final clean shape: (16, 248, 4000)
Task: task_story_math      | Final clean shape: (16, 248, 4000)
Task: task_working_memory  | Final clean shape: (16, 248, 4000)


<div class="alert alert-block alert-success">
Model
</div>

In [3]:
tasks = ['rest', 'task_motor', 'task_story_math', 'task_working_memory']

In [75]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class MEGClassifierCNN(nn.Module):
    #CNN class
    def __init__(self, filters, dropout):
        super().__init__() #herits from nn.Module
        #input size 248x4000
        #first convolution layer, 1 Dimension for the 248x1 input, stride 2 to divide the output size
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=248, out_channels=filters//4, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(filters//4), #normalisation
            nn.ReLU(), #relu for activation
            nn.MaxPool1d(kernel_size=4) #pooling to reduce temporal dimension
        ) 
        #size 64x500
        #second convolution layer
        self.conv2 = nn.Sequential(
            nn.Conv1d(in_channels=filters//4, out_channels=filters//2, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(filters//2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4)
        )
        #size 128x62
        #third convolution layer
        self.conv3 = nn.Sequential(
            nn.Conv1d(in_channels=filters//2, out_channels=filters, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(filters),
            nn.ReLU()
        )
        #size 256x62
        #global pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        #size 256x1
        #fully connected layer 
        self.fc = nn.Sequential(
            nn.Linear(filters, filters//4),   #first compression, 256 to 64 values
            nn.ReLU(),
            nn.Dropout(dropout),      #70% neuron dropout during training
            nn.Linear(filters//4, 4)      #second compression, 64 to 4 output values (the 4 states)
        )
        #final size 4x1


    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1) #flattening
        x = self.fc(x)
        return x

In [5]:
class MEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

<div class="alert alert-block alert-danger">
Data processing
</div>

In [6]:
def downsampling_data(data, nr_sample):
    data_downsampled = signal.resample(data, num = nr_sample, axis = 1)

    return  data_downsampled

In [7]:

def train_data_processing(set):
    if set == "Intra": 
        nrs = ["_105923_"]
        file_dir = r'/Users/amancedennery/Downloads/Final Project data/Intra/train'
    else : 
        nrs = ["_113922_",  "_164636_"]
        file_dir = r'/Users/amancedennery/Downloads/Final Project data/Cross/train'
        
    ini_arrays = {'rest': [], 'task_motor': [], 'task_story_math': [], 
                  'task_working_memory': []}    

    for task in tasks:
        for nr in nrs:
            for i in range(1, 9):
                #print(task+nr+str(i)+'.h5')
                file_path = os.path.join(file_dir, task+nr+str(i)+'.h5')
                

                with h5py.File(file_path, 'r')as f:
                    file_keys = list(f.keys())
                    if not file_keys:
                            print(f"Warning: File {file_path} is empty inside!")
                            continue
                    
                    dataset01 = file_keys[0]
                    
                    matrix = f.get(dataset01)[()]
                    # print(type(matrix))
                    # print(matrix.shape)
                    ini_arrays[task].append(matrix)        

    downsampled_dict = {}
    all_data_for_stats = []
    for task in tasks:
        raw_files = ini_arrays[task]
        downsampled_files = [downsampling_data(f, 4000) for f in raw_files]
        downsampled_dict[task] = downsampled_files
        # On ajoute TOUS les fichiers pour calculer les stats globales
        for ds_file in downsampled_files:
            all_data_for_stats.append(ds_file)
    # Stats globales sur TOUTES les tâches
    all_data_concat = np.concatenate(all_data_for_stats, axis=1)  # (248, très_grand)
    global_mean = np.mean(all_data_concat, axis=1, keepdims=True)  # (248, 1)
    global_std = np.std(all_data_concat, axis=1, keepdims=True)    # (248, 1)
    global_std = np.where(global_std == 0, 1.0, global_std)
    # Normalisation avec les stats globales
    dict_processed = {}
    for task in tasks:
        normalized_files = []
        for ds_matrix in downsampled_dict[task]:
            norm_matrix = (ds_matrix - global_mean) / global_std
            normalized_files.append(norm_matrix)
        dict_processed[task] = np.array(normalized_files)
        print(f"Task: {task:<20} | Final clean shape: {dict_processed[task].shape}")

    return dict_processed, global_mean, global_std

In [8]:
data_intra_train, global_mean_intra, global_std_intra = train_data_processing("Intra")

Task: rest                 | Final clean shape: (8, 248, 4000)
Task: task_motor           | Final clean shape: (8, 248, 4000)
Task: task_story_math      | Final clean shape: (8, 248, 4000)
Task: task_working_memory  | Final clean shape: (8, 248, 4000)


In [9]:
# stats = {task: {'sum': np.zeros((248, 1)), 'sq_sum': np.zeros((248, 1)), 'steps': 0} for task in tasks}

def intra_test_data_processing(global_mean, global_std):
    file_dir = r'/Users/amancedennery/Downloads/Final Project data/Intra/test/'
    nr = '_105923_'
    ini_arrays = {'rest': [], 'task_motor': [], 'task_story_math': [], 
                  'task_working_memory': []}    

    for task in tasks:
        for i in [9, 10]:
            #print(task+nr+str(i)+'.h5')
            file_path = os.path.join(file_dir, task+nr+str(i)+'.h5')
            

            with h5py.File(file_path, 'r')as f:
                file_keys = list(f.keys())
                if not file_keys:
                        print(f"Warning: File {file_path} is empty inside!")
                        continue
                
                dataset01 = file_keys[0]
                
                matrix = f.get(dataset01)[()]
                # print(type(matrix))
                # print(matrix.shape)
                ini_arrays[task].append(matrix)



    processed_datasets_intra_test = {}
    for task_arr in ini_arrays:
        raw_files = ini_arrays[task_arr]
        downsampled_files = [downsampling_data(f, 4000) for f in raw_files]

        normalized_files = []
        for ds_matrix in downsampled_files:
            norm_matrix = (ds_matrix - global_mean) / global_std
            normalized_files.append(norm_matrix)

        processed_datasets_intra_test[task_arr] = np.array(normalized_files)
        
        print(f"Task: {task_arr:<20} | Final clean shape: {processed_datasets_intra_test[task_arr].shape}")
    return processed_datasets_intra_test



In [10]:
X_list = []
y_list = []

for label, task in enumerate(tasks):
    data = data_intra_train[task]   
    X_list.append(data)
    y_list.append(np.full(data.shape[0], label))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

train_intra_dataset = MEGDataset(X, y)
train_intra_loader = DataLoader(train_intra_dataset, batch_size=8, shuffle=True)

In [12]:
intra_test_data = intra_test_data_processing(global_mean_intra, global_std_intra)

X_list = []
y_list = []

for label, task in enumerate(tasks):
    data = intra_test_data[task]   
    X_list.append(data)
    y_list.append(np.full(data.shape[0], label))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

test_intra_dataset = MEGDataset(X, y)
test_intra_loader = DataLoader(test_intra_dataset, batch_size=1, shuffle=False)


Task: rest                 | Final clean shape: (2, 248, 4000)
Task: task_motor           | Final clean shape: (2, 248, 4000)
Task: task_story_math      | Final clean shape: (2, 248, 4000)
Task: task_working_memory  | Final clean shape: (2, 248, 4000)


<div class="alert alert-block alert-warning">
Training and testing
</div>

In [73]:
def train_model(data, filters, dropout, num_epochs):
    model = MEGClassifierCNN(filters, dropout)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    epoch = 0
    patience = 0
    #prev = -1.0
    while (epoch<num_epochs):# and (patience < 5):

        model.train() #train mode, dropout activated

        loss = 0.0
        correct = 0
        total = 0

        for batch_X, batch_y in data:
            
            pred = model(batch_X) #predictions
            loss_ = criterion(pred, batch_y) #computes error

            optimizer.zero_grad() #gradients renitialized
            loss_.backward() #computes gradients 
            optimizer.step() #updates weights

            loss += loss_.item()
            _, predicted_classes = torch.max(pred, 1)
            total += batch_y.size(0)
            correct += (predicted_classes == batch_y).sum().item()
            
        epoch_loss = loss / len(data)
        """if prev == -1.0:
            prev = epoch_loss
        else :
            if epoch_loss-prev>0.0:
                patience += 1
            else : 
                if patience > 0:
                    patience -= 2
                else :
                    patience = 0
            prev = epoch_loss"""
        epoch_acc = 100 * correct / total
        epoch +=1
        if (epoch + 1) % 2 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.1f}%")
    return model

In [ ]:
model_intra = train_model(train_intra_loader)

Epoch [2/30] | Loss: 1.3719 | Accuracy: 25.0%
Epoch [4/30] | Loss: 1.0639 | Accuracy: 62.5%
Epoch [6/30] | Loss: 0.9589 | Accuracy: 78.1%
Epoch [8/30] | Loss: 0.7343 | Accuracy: 81.2%
Epoch [10/30] | Loss: 0.5273 | Accuracy: 93.8%
Epoch [12/30] | Loss: 0.6699 | Accuracy: 84.4%
Epoch [14/30] | Loss: 0.5967 | Accuracy: 81.2%
Epoch [16/30] | Loss: 0.4149 | Accuracy: 90.6%
Epoch [18/30] | Loss: 0.2459 | Accuracy: 96.9%
Epoch [20/30] | Loss: 0.3767 | Accuracy: 87.5%
Epoch [22/30] | Loss: 0.2724 | Accuracy: 93.8%
Epoch [24/30] | Loss: 0.2767 | Accuracy: 93.8%
Epoch [26/30] | Loss: 0.3637 | Accuracy: 87.5%
Epoch [28/30] | Loss: 0.2636 | Accuracy: 90.6%
Epoch [30/30] | Loss: 0.2794 | Accuracy: 90.6%


In [55]:
def test_model(data, model):
    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    correct = 0
    total = len(data)
    i = 0 

    model.eval()
    with torch.no_grad():
        for batch_X, batch_y in data:
        
            pred = model(batch_X) #predictions
            loss_ = criterion(pred, batch_y) #computes error

            total_loss += loss_.item()
            _, predicted_classes = torch.max(pred, 1)
            
            correct += (predicted_classes == batch_y).sum().item()
            print(f"Prediction [{i+1}/{total}] | Loss: {loss_.item():.4f} | Class : {batch_y} | Predicted class : {predicted_classes}")

            i+=1
        acc = 100 * correct / total
        print(f"Avg Loss: {total_loss/total:.4f} | Accuracy: {acc:.1f}%")

In [ ]:
test_model(test_intra_loader, model_intra)

tensor([[ 2.5304, -0.9199, -2.2398, -1.2398]])
Prediction [1/8] | Loss: 0.0613 | Class : tensor([0]) | Predicted class : tensor([0])
tensor([[ 2.6285, -0.8816, -2.3499, -1.3772]])
Prediction [2/8] | Loss: 0.0535 | Class : tensor([0]) | Predicted class : tensor([0])
tensor([[-2.7246,  4.7016, -2.5139, -3.4266]])
Prediction [3/8] | Loss: 0.0016 | Class : tensor([1]) | Predicted class : tensor([1])
tensor([[-2.9086,  5.0157, -2.6910, -3.3786]])
Prediction [4/8] | Loss: 0.0010 | Class : tensor([1]) | Predicted class : tensor([1])
tensor([[-1.8689, -1.1331,  2.4151, -1.9494]])
Prediction [5/8] | Loss: 0.0538 | Class : tensor([2]) | Predicted class : tensor([2])
tensor([[-1.8702, -1.1367,  2.3971, -1.9049]])
Prediction [6/8] | Loss: 0.0552 | Class : tensor([2]) | Predicted class : tensor([2])
tensor([[-1.5972, -3.1571, -2.7245,  4.3518]])
Prediction [7/8] | Loss: 0.0040 | Class : tensor([3]) | Predicted class : tensor([3])
tensor([[-1.5793, -3.1480, -2.7186,  4.3319]])
Prediction [8/8] | Los

<div class="alert alert-block alert-info">
Cross
</div>

In [16]:
data_cross_train, global_mean_cross, global_std_cross = train_data_processing("Cross")

Task: rest                 | Final clean shape: (16, 248, 4000)
Task: task_motor           | Final clean shape: (16, 248, 4000)
Task: task_story_math      | Final clean shape: (16, 248, 4000)
Task: task_working_memory  | Final clean shape: (16, 248, 4000)


In [50]:
X_list = []
y_list = []

for label, task in enumerate(tasks):
    data = data_cross_train[task]   
    X_list.append(data)
    y_list.append(np.full(data.shape[0], label))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

train_cross_dataset = MEGDataset(X, y)
train_cross_loader = DataLoader(train_cross_dataset, batch_size=8, shuffle=True)

In [105]:
model_cross = train_model(train_cross_loader, 128, 0.5, 75)

Epoch [2/75] | Loss: 1.2952 | Accuracy: 39.1%
Epoch [4/75] | Loss: 0.8396 | Accuracy: 84.4%
Epoch [6/75] | Loss: 0.6935 | Accuracy: 84.4%
Epoch [8/75] | Loss: 0.5136 | Accuracy: 87.5%
Epoch [10/75] | Loss: 0.3302 | Accuracy: 93.8%
Epoch [12/75] | Loss: 0.4182 | Accuracy: 87.5%
Epoch [14/75] | Loss: 0.2690 | Accuracy: 95.3%
Epoch [16/75] | Loss: 0.1842 | Accuracy: 98.4%
Epoch [18/75] | Loss: 0.3338 | Accuracy: 93.8%
Epoch [20/75] | Loss: 0.4298 | Accuracy: 84.4%
Epoch [22/75] | Loss: 0.2045 | Accuracy: 96.9%
Epoch [24/75] | Loss: 0.5047 | Accuracy: 78.1%
Epoch [26/75] | Loss: 0.1676 | Accuracy: 98.4%
Epoch [28/75] | Loss: 0.2220 | Accuracy: 92.2%
Epoch [30/75] | Loss: 0.1369 | Accuracy: 96.9%
Epoch [32/75] | Loss: 0.2841 | Accuracy: 93.8%
Epoch [34/75] | Loss: 0.1334 | Accuracy: 98.4%
Epoch [36/75] | Loss: 0.1419 | Accuracy: 96.9%
Epoch [38/75] | Loss: 0.0854 | Accuracy: 98.4%
Epoch [40/75] | Loss: 0.1134 | Accuracy: 92.2%
Epoch [42/75] | Loss: 0.0649 | Accuracy: 100.0%
Epoch [44/75] | 

In [64]:
def cross_test_data_processing(global_mean, global_std):
    file_dir = r'/Users/amancedennery/Downloads/Final Project data/Cross/test'
    nrs = {1:["_162935_"], 2:["_707749_"], 3:["_725751_", "_735148_"]}

    all_data = dict()
    for j in [1, 2, 3]:
        ini_arrays = {'rest': [], 'task_motor': [], 'task_story_math': [], 'task_working_memory': []}    
        for task in tasks:
            for nr in nrs[j]:
                for i in range(11):
                    file_path = os.path.join(file_dir+str(j), task+nr+str(i)+'.h5')
                    try :
                        with h5py.File(file_path, 'r')as f:
                            file_keys = list(f.keys())
                            if not file_keys or not file_keys[0]:
                                    print(f"Warning: File {file_path} is empty inside!")
                                    continue
                                
                            dataset01 = file_keys[0]
                            matrix = f.get(dataset01)[()]
                            ini_arrays[task].append(matrix)
                    except FileNotFoundError:
                        continue

        processed_datasets_cross_test = {}
        for task_arr in ini_arrays:
            raw_files = ini_arrays[task_arr]
            downsampled_files = [downsampling_data(f, 4000) for f in raw_files]

            normalized_files = []
            for ds_matrix in downsampled_files:
                norm_matrix = (ds_matrix - global_mean) / global_std
                normalized_files.append(norm_matrix)

            processed_datasets_cross_test[task_arr] = np.array(normalized_files)
            print(f"Task: {task_arr:<20} | Final clean shape: {processed_datasets_cross_test[task_arr].shape}")
        all_data[j]=(processed_datasets_cross_test)

    return all_data



In [63]:
cross_test_data = cross_test_data_processing(global_mean_cross, global_std_cross)

Task: rest                 | Final clean shape: (4, 248, 4000)
Task: task_motor           | Final clean shape: (4, 248, 4000)
Task: task_story_math      | Final clean shape: (4, 248, 4000)
Task: task_working_memory  | Final clean shape: (4, 248, 4000)
1
Task: rest                 | Final clean shape: (4, 248, 4000)
Task: task_motor           | Final clean shape: (4, 248, 4000)
Task: task_story_math      | Final clean shape: (4, 248, 4000)
Task: task_working_memory  | Final clean shape: (4, 248, 4000)
2
Task: rest                 | Final clean shape: (4, 248, 4000)
Task: task_motor           | Final clean shape: (4, 248, 4000)
Task: task_story_math      | Final clean shape: (4, 248, 4000)
Task: task_working_memory  | Final clean shape: (4, 248, 4000)
3


<div class="alert alert-block alert-warning">
Cross - Test 1
</div>

In [82]:
X_list = []
y_list = []

for label, task in enumerate(tasks):
    data = cross_test_data[1][task]   
    X_list.append(data)
    y_list.append(np.full(data.shape[0], label))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

test1_cross_dataset = MEGDataset(X, y)
test1_cross_loader = DataLoader(test1_cross_dataset, batch_size=1, shuffle=False)


In [106]:
test_model(test1_cross_loader, model_cross)

Prediction [1/16] | Loss: 0.0420 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [2/16] | Loss: 0.8460 | Class : tensor([0]) | Predicted class : tensor([3])
Prediction [3/16] | Loss: 4.7155 | Class : tensor([0]) | Predicted class : tensor([3])
Prediction [4/16] | Loss: 1.1129 | Class : tensor([0]) | Predicted class : tensor([3])
Prediction [5/16] | Loss: 0.0021 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [6/16] | Loss: 0.0047 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [7/16] | Loss: 0.0040 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [8/16] | Loss: 0.0112 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [9/16] | Loss: 0.0032 | Class : tensor([2]) | Predicted class : tensor([2])
Prediction [10/16] | Loss: 0.0001 | Class : tensor([2]) | Predicted class : tensor([2])
Prediction [11/16] | Loss: 0.0004 | Class : tensor([2]) | Predicted class : tensor([2])
Prediction [12/16] | Loss: 0.0001 | Class

<div class="alert alert-block alert-warning">
Cross - Test 2
</div>

In [84]:
X_list = []
y_list = []

for label, task in enumerate(tasks):
    data = cross_test_data[2][task]   
    X_list.append(data)
    y_list.append(np.full(data.shape[0], label))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

test2_cross_dataset = MEGDataset(X, y)
test2_cross_loader = DataLoader(test2_cross_dataset, batch_size=1, shuffle=False)


In [107]:
test_model(test2_cross_loader, model_cross)

Prediction [1/16] | Loss: 0.0048 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [2/16] | Loss: 0.0013 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [3/16] | Loss: 0.0005 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [4/16] | Loss: 0.0142 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [5/16] | Loss: 9.2151 | Class : tensor([1]) | Predicted class : tensor([2])
Prediction [6/16] | Loss: 15.8488 | Class : tensor([1]) | Predicted class : tensor([2])
Prediction [7/16] | Loss: 16.6501 | Class : tensor([1]) | Predicted class : tensor([2])
Prediction [8/16] | Loss: 18.2105 | Class : tensor([1]) | Predicted class : tensor([2])
Prediction [9/16] | Loss: 9.1465 | Class : tensor([2]) | Predicted class : tensor([1])
Prediction [10/16] | Loss: 12.9165 | Class : tensor([2]) | Predicted class : tensor([1])
Prediction [11/16] | Loss: 16.3152 | Class : tensor([2]) | Predicted class : tensor([1])
Prediction [12/16] | Loss: 25.4116 |

In [96]:
X_list = []
y_list = []

for label, task in enumerate(tasks):
    data = cross_test_data[3][task]   
    X_list.append(data)
    y_list.append(np.full(data.shape[0], label))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

test3_cross_dataset = MEGDataset(X, y)
test3_cross_loader = DataLoader(test3_cross_dataset, batch_size=1, shuffle=False)


In [108]:
test_model(test3_cross_loader, model_cross)

Prediction [1/16] | Loss: 0.0003 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [2/16] | Loss: 0.0000 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [3/16] | Loss: 0.0112 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [4/16] | Loss: 0.0027 | Class : tensor([0]) | Predicted class : tensor([0])
Prediction [5/16] | Loss: 0.0004 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [6/16] | Loss: 0.0001 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [7/16] | Loss: 0.0025 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [8/16] | Loss: 0.0010 | Class : tensor([1]) | Predicted class : tensor([1])
Prediction [9/16] | Loss: 0.0018 | Class : tensor([2]) | Predicted class : tensor([2])
Prediction [10/16] | Loss: 0.0179 | Class : tensor([2]) | Predicted class : tensor([2])
Prediction [11/16] | Loss: 0.0008 | Class : tensor([2]) | Predicted class : tensor([2])
Prediction [12/16] | Loss: 0.0008 | Class